# 02 — Softmax and cross-entropy over GPT-2's 50,257 tokens

At every position, GPT-2 small outputs 50,257 logits, one score per token in the vocabulary. Softmax turns them into a probability distribution, and cross-entropy measures how much probability went to the correct next token.

The backward pass starts at the loss. The very first gradient, $\partial\mathcal{L}/\partial\,\text{logits}$, comes from cross-entropy, and everything else in the network is that signal flowing backwards. Softmax also appears *inside* attention, so we need its backward pass on its own as well.

In this notebook:

1. softmax forward and why we subtract the max;
2. the softmax Jacobian, its vector–Jacobian product, and the intuition behind it;
3. cross-entropy via log-softmax / logsumexp;
4. the famous gradient `(probs − onehot) / N` and what it means;
5. the exact initial loss to expect from GPT-2 small's initialization (≈ 10.98, not quite ln 50257 = 10.82);
6. z-loss (since your script has a stub for it).

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True, linewidth=120)

def numerical_grad(f, x, eps=1e-6):
    grad = np.zeros_like(x, dtype=np.float64)
    it = np.nditer(x, flags=["multi_index"])
    while not it.finished:
        i = it.multi_index
        old = x[i]
        x[i] = old + eps; fp = f()
        x[i] = old - eps; fm = f()
        x[i] = old
        grad[i] = (fp - fm) / (2 * eps)
        it.iternext()
    return grad

def rel_error(a, b):
    return np.linalg.norm(a - b) / max(1e-12, np.linalg.norm(a) + np.linalg.norm(b))

def spot_check(f, p, g, k=20, eps=1e-5, seed=123):
    """Check k random entries of the analytic gradient g (of scalar f() w.r.t. array p) against
    central differences. Entries with near-zero gradient are skipped: there the finite difference
    is pure rounding noise, so a relative error would be meaningless."""
    r = np.random.default_rng(seed)
    candidates = np.flatnonzero(np.abs(g) > 1e-3 * np.abs(g).max())
    worst = 0.0
    for i in r.choice(candidates, size=min(k, candidates.size), replace=False):
        ii = np.unravel_index(i, p.shape)
        old = p[ii]
        p[ii] = old + eps; fp = f()
        p[ii] = old - eps; fm = f()
        p[ii] = old
        num = (fp - fm) / (2 * eps)
        worst = max(worst, abs(num - g[ii]) / (abs(num) + abs(g[ii])))
    return worst

rng = np.random.default_rng(0)

## 1. Softmax forward

$$
s_i = \frac{e^{z_i}}{\sum_k e^{z_k}}
$$

This is applied along the **last axis**: over the 50,257 vocabulary entries for logits `(B, T, 50257)`, and over key positions for attention scores `(B, 12, T, T)`.

**Stability.** $e^{1000}$ overflows to `inf`. But softmax is invariant to adding a constant to all inputs:

$$
\frac{e^{z_i + c}}{\sum_k e^{z_k + c}} = \frac{e^{c}e^{z_i}}{e^{c}\sum_k e^{z_k}} = s_i,
$$

so we subtract $c=\max_k z_k$. The largest exponent becomes $e^0=1$, nothing overflows, and at least one term of the denominator is 1, so it never underflows to 0 either. Your script's forward pass does this correctly. It just computes `np.exp(...)` twice, which you can avoid by storing it.

## 2. Softmax backward

### The Jacobian

Using the quotient rule on $s_i = e^{z_i}/S$ with $S=\sum_k e^{z_k}$:

- for $j = i$: $\dfrac{\partial s_i}{\partial z_i} = \dfrac{e^{z_i}S - e^{z_i}e^{z_i}}{S^2} = s_i - s_i^2$;
- for $j \neq i$: $\dfrac{\partial s_i}{\partial z_j} = \dfrac{0\cdot S - e^{z_i}e^{z_j}}{S^2} = -s_i s_j$.

Combined: $\;\dfrac{\partial s_i}{\partial z_j} = s_i(\delta_{ij} - s_j)$, i.e. $J = \mathrm{diag}(s) - s s^\top$. (It's symmetric.)

### The vector–Jacobian product

Given the upstream gradient $ds$:

$$
dz_j = \sum_i ds_i\, s_i(\delta_{ij} - s_j) = s_j\,ds_j - s_j\sum_i ds_i s_i
\quad\Longrightarrow\quad
\boxed{dz = s \odot \big(ds - \langle ds, s\rangle\big)}
$$

where $\langle ds, s\rangle = \sum_i ds_i s_i$ is a scalar per row. This costs $O(V)$ per row instead of the $O(V^2)$ of a Jacobian.

### Intuition

- $\langle ds, s\rangle$ is the **probability-weighted average** of the upstream gradient. Each logit's gradient is "how much worse than average is the upstream gradient for my output", times my probability.
- **The gradient always sums to zero:** $\sum_j dz_j = \langle s, ds\rangle - \langle ds, s\rangle\sum_j s_j = 0$. That must happen: adding the same constant to every logit doesn't change the output, so the loss can't care about the $(1,1,\dots,1)$ direction. It's the same "project out the invariant direction" story you saw in the LayerNorm gradient (there it was two directions: mean and scale).
- Entries with $s_j \approx 0$ get almost no gradient. In attention, **masked positions have $s_j = 0$ exactly, so they receive exactly zero gradient.** You don't need any special masking logic in the backward pass.

In [ ]:
class Softmax:
    def forward(self, x):
        z = x - x.max(axis=-1, keepdims=True)
        e = np.exp(z)
        self.s = e / e.sum(axis=-1, keepdims=True)   # cache the OUTPUT; backward only needs s
        return self.s

    def backward(self, dout):
        s = self.s
        return s * (dout - (dout * s).sum(axis=-1, keepdims=True))

# check on a small attention-shaped input (B, H, T, T)
sm = Softmax()
z = rng.normal(size=(2, 2, 3, 3))
G = rng.normal(size=z.shape)
f = lambda: np.sum(sm.forward(z) * G)
f()
dz = sm.backward(G)
print("softmax dz rel err:", rel_error(dz, numerical_grad(f, z)))
print("row sums of dz (should be ~0):", dz.sum(-1).ravel()[:6])

## 3. Cross-entropy forward

For row $n$ with logits $z_n\in\mathbb{R}^V$ and correct class $y_n$:

$$
\ell_n = -\log s_{n,y_n} = -\big(z_{n,y_n} - \operatorname{lse}(z_n)\big),
\qquad
\operatorname{lse}(z) = \log\sum_k e^{z_k} = m + \log\sum_k e^{z_k - m},\; m=\max_k z_k .
$$

$\mathcal{L} = \frac1N\sum_n \ell_n$, with $N=B\cdot T$.

Why compute it via **logsumexp** instead of `-np.log(softmax(z)[y])`? If the model assigns a tiny probability to the correct token, `softmax` can underflow to exactly 0 and `log(0) = -inf`. The log-softmax form $z_y - \operatorname{lse}(z)$ never exponentiates and logs separately, so it stays finite. Your script's forward pass does exactly this. 👍

## 4. Cross-entropy backward: `(probs − onehot) / N`

Differentiate $\ell = -z_y + \operatorname{lse}(z)$ with respect to $z_j$:

- $\partial(-z_y)/\partial z_j = -\delta_{jy}$
- $\partial\operatorname{lse}(z)/\partial z_j = \dfrac{e^{z_j}}{\sum_k e^{z_k}} = s_j$ (the derivative of logsumexp *is* softmax)

$$
\frac{\partial \ell_n}{\partial z_{n,j}} = s_{n,j} - \mathbb{1}[j = y_n]
\qquad\Longrightarrow\qquad
\boxed{d\,\text{logits} = \frac{1}{N}\big(\text{probs} - \text{onehot}(y)\big)}
$$

The $1/N$ comes from the mean. This is the starting `dout` handed to `model.backward(...)`.

### Intuition: a tug of war with a fixed budget

Gradient descent moves logits *against* the gradient:

- the **correct** logit has gradient $s_y - 1 \le 0$, so it gets pushed **up**, harder the less probability it currently has;
- every **wrong** logit $j$ has gradient $s_j \ge 0$, so it gets pushed **down** in proportion to how much probability it is "stealing";
- each row sums to 0 (same reason as softmax): total push up = total push down = $1 - s_y$.

When the model is already confident and right ($s_y \to 1$), the whole row goes to 0 and that example stops teaching anything. Learning concentrates on what the model gets wrong.

In [ ]:
class CrossEntropyLoss:
    def forward(self, logits, targets):
        B, T, V = logits.shape
        N = B * T
        l = logits.reshape(N, V)
        y = targets.reshape(N)
        m = l.max(axis=1, keepdims=True)
        lse = m + np.log(np.exp(l - m).sum(axis=1, keepdims=True))   # (N, 1)
        logp = l - lse                                                # log-softmax, (N, V)
        self.probs = np.exp(logp)                                     # cache softmax for backward
        self.y, self.shape = y, (B, T, V)
        return -logp[np.arange(N), y].mean()

    def backward(self):
        N = self.y.shape[0]
        d = self.probs.copy()
        d[np.arange(N), self.y] -= 1.0      # probs - onehot, without building the onehot matrix
        d /= N
        return d.reshape(self.shape)        # (B, T, V)

B, T, V = 2, 3, 5
logits = rng.normal(size=(B, T, V))
targets = rng.integers(0, V, size=(B, T))
ce = CrossEntropyLoss()
f = lambda: ce.forward(logits, targets)
f()
dl = ce.backward()
print("CE dlogits rel err:", rel_error(dl, numerical_grad(f, logits)))
print("one row of dlogits * N:", dl[0, 0] * B * T, " target =", targets[0, 0])

### At GPT-2's real vocabulary size

The same check with V = 50,257, spot-checking random entries (B=2, T=4, so 8 × 50,257 logits):

In [ ]:
V_gpt2 = 50257
logits_big = rng.normal(size=(2, 4, V_gpt2))
targets_big = rng.integers(0, V_gpt2, size=(2, 4))
f = lambda: ce.forward(logits_big, targets_big)
f()
dl_big = ce.backward()
print("dlogits spot rel err:", spot_check(f, logits_big, dl_big))
print("each row sums to ~0:", np.abs(dl_big.sum(-1)).max())

### Why fuse softmax and cross-entropy?

You *could* treat them as two layers: softmax, then $\ell = -\log s_y$ with $\partial\ell/\partial s_y = -1/s_y$. Then run the softmax backward pass. Let's verify it gives the same answer:

$$
dz_j = s_j\big(ds_j - \langle ds,s\rangle\big),\quad ds = -\tfrac{1}{s_y}e_y,\quad \langle ds,s\rangle = -1
\;\Longrightarrow\; dz_j = s_j\big(-\tfrac{1}{s_y}\delta_{jy} + 1\big) = s_j - \delta_{jy}. ✓
$$

The math agrees, but the two-layer route computes $1/s_y$, which blows up when $s_y$ is tiny, and then multiplies it by $s_y$ again. The fused formula never divides. That's why every framework fuses them, and why your class is named `CrossEntropyWithSoftmax`.

In [ ]:
# numerical demonstration that the two-layer route agrees (on a well-behaved example)
sm = Softmax()
s = sm.forward(logits.reshape(-1, V))
ds = np.zeros_like(s)
ds[np.arange(B * T), targets.ravel()] = -1.0 / s[np.arange(B * T), targets.ravel()] / (B * T)
print("two-layer route vs fused:", rel_error(sm.backward(ds).reshape(B, T, V), dl))

## 5. The initial loss of GPT-2 small: ≈ 10.98

If the model's logits were all equal, the distribution would be uniform, $s_j = 1/V$, and the loss would be $-\log(1/V) = \ln V = \ln 50257 \approx 10.82$. That is the loss of a model that knows nothing.

GPT-2's initialized logits are small but not exactly zero. The output head computes $z_j = x_f \cdot w_j$, where $x_f$ is the final LayerNorm's output (768 entries with mean 0 and variance 1) and $w_j$ is row $j$ of `wte` (entries drawn from $\mathcal{N}(0, 0.02^2)$). So each logit is roughly Gaussian with variance

$$
s^2 = 768 \times 0.02^2 = 0.3072 .
$$

The loss is $\operatorname{lse}(z) - z_y$. On average $z_y$ is 0. For the logsumexp, use the mean of a log-normal variable, $\mathbb{E}[e^{z}] = e^{s^2/2}$:

$$
\operatorname{lse}(z) = \log\sum_{j=1}^{V} e^{z_j} \approx \log\big(V\, e^{s^2/2}\big) = \ln V + \frac{s^2}{2}
\quad\Longrightarrow\quad
\mathcal{L}_0 \approx 10.825 + 0.154 = 10.98 .
$$

**This is the first check to run on any new model.** If your first logged loss is far from this (say 15, or 50), something is wrong before training starts: bad initialization, a missing LayerNorm, wrong scaling, a broken mask.

In [ ]:
V, C = 50257, 768
wte = rng.normal(0, 0.02, size=(V, C)).astype(np.float32)          # GPT-2 small's embedding init
x_f = rng.normal(size=(4, 32, C)).astype(np.float32)               # stand-in for ln_f output
x_f = (x_f - x_f.mean(-1, keepdims=True)) / x_f.std(-1, keepdims=True)   # mean 0, variance 1 per token
logits = x_f @ wte.T                                               # (4, 32, 50257)
loss0 = ce.forward(logits, rng.integers(0, V, size=(4, 32)))
print(f"logit std {logits.std():.3f} (predicted {np.sqrt(C) * 0.02:.3f})")
print(f"initial loss {loss0:.3f}   predicted ln V + s^2/2 = {np.log(V) + C * 0.02**2 / 2:.3f}   ln V = {np.log(V):.3f}")

### Memory: the logits are the largest tensor in GPT-2

During training the loss layer holds three tensors of shape `(B, T, 50257)`: the logits, the cached probabilities, and `dlogits`. At B = 8, T = 1024 in float32, each is 8 × 1024 × 50257 × 4 bytes ≈ **1.6 GB**. (Production implementations compute the loss in chunks or with a fused kernel to avoid holding all three.) In notebook 06 we train with small B and T for exactly this reason.

## 6. Bonus: z-loss (your `Z_lossWithSoftmax` stub)

GPT-2 does **not** use this, but you have a stub for it, so here it is. Z-loss (from PaLM) adds a penalty on the size of the log-normalizer:

$$
\mathcal{L}_z = \alpha\cdot\frac1N\sum_n \operatorname{lse}(z_n)^2 .
$$

It keeps logits from drifting to huge values, which improves numerical stability in low precision. Its gradient follows directly from $\partial\operatorname{lse}/\partial z = s$:

$$
\frac{\partial\mathcal{L}_z}{\partial z_{n,j}} = \frac{2\alpha}{N}\operatorname{lse}(z_n)\, s_{n,j}.
$$

You add this to the cross-entropy gradient: **when two losses are summed, their gradients are summed** (rule 2 again).

In [ ]:
class ZLoss:
    def __init__(self, alpha=1e-4):
        self.alpha = alpha
    def forward(self, logits):
        V = logits.shape[-1]
        l = logits.reshape(-1, V)
        m = l.max(axis=1, keepdims=True)
        e = np.exp(l - m)
        self.lse = m + np.log(e.sum(axis=1, keepdims=True))
        self.s = e / e.sum(axis=1, keepdims=True)
        self.shape = logits.shape
        return self.alpha * np.mean(self.lse**2)
    def backward(self):
        N = self.lse.shape[0]
        return (2 * self.alpha / N * self.lse * self.s).reshape(self.shape)

zl = ZLoss(alpha=0.1)
small_logits = rng.normal(size=(2, 3, 5))
f = lambda: zl.forward(small_logits)
f()
print("z-loss rel err:", rel_error(zl.backward(), numerical_grad(f, small_logits)))

## Summary

| | Forward | Backward |
|---|---|---|
| Softmax | `s = exp(z - max) / sum` | `dz = s * (ds - sum(ds*s, -1))` |
| Cross-entropy (fused) | `-mean(z_y - lse(z))` | `dlogits = (probs - onehot) / N` |

- The derivative of logsumexp is softmax. That fact produces both results.
- Softmax and cross-entropy gradients sum to zero along the last axis (shift invariance).
- Masked attention positions get zero gradient automatically.
- First check for any new model: the initial loss. For GPT-2 small, expect ≈ 10.98 ($\ln 50257 + 768\cdot 0.02^2/2$).

Next: **03 — LayerNorm**, the backward pass you've already seen in theory, now implemented and checked on `(B,T,C)` tensors.